# L09 · TCOD

## Goal

**예상 시간:** 40분 · **경로:** full

- 시간 깊이 schedule을 만든다
- F2B/B2F를 구분한다
- 선택된 turn을 감사한다

### 현재 위치: L08 → **L09** → L10

```text
Prompt/Data -> state source -> ... -> L09 -> ... -> fair evaluation
```

Alt text: The course map highlights L09 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L09"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L09', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

TCOD는 처음부터 긴 trajectory 전체를 증류하지 않는다. F2B는 앞에서부터 깊이를 늘리고, B2F는 teacher/successful prefix 뒤의 마무리부터 student에게 넘긴다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

TCOD는 긴 multi-turn trajectory를 한 번에 맡기지 않고 student가 담당할 시간 구간을 점차 늘린다. F2B는 앞 turn부터 늘려 초기 의사결정을 연습한다. B2F는 성공한 teacher prefix 뒤의 마지막 turn부터 맡겨 쉬운 suffix에서 시작한다.

curriculum depth와 pacing은 별개다. depth는 현재 포함할 turn 수, pacing은 몇 optimizer step마다 depth를 늘릴지다. 잘린 trajectory에서도 observation/action 경계와 loss mask가 원래 turn과 정렬되어야 한다.

### 실제 구현: 왜 이렇게 만들었나

`curriculum_depth`는 step과 pacing을 정수식으로 계산한다. `temporal_curriculum_mask`는 F2B/B2F가 선택한 turn과 원 response mask의 교집합만 반환한다. B2F의 teacher-prefix 생성은 paper-scale 인프라 대신 mini backend에서 명시적 근사다.

실제 코드: [`tcod.py`](../../src/opd_study/algorithms/tcod.py), [`advanced.py`](../../src/opd_study/training/advanced.py).

In [2]:
import inspect
from opd_study.algorithms.tcod import temporal_curriculum_mask, tcod_loss

objects_to_show = (temporal_curriculum_mask, tcod_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.tcod.temporal_curriculum_mask
def temporal_curriculum_mask(
    trajectories: TrajectoryBatch, *, depth: int, direction: str
) -> Tensor:
    """Select early F2B or late B2F turns from a correctly sourced trajectory.

    For B2F, callers must supply trajectories whose earlier history came from the
    teacher/successful prefix.  A mask alone cannot manufacture that state distribution.
    """

    if trajectories.turn_ids is None:
        raise ValueError("TCOD requires turn_ids")
    if depth < 1:
        raise ValueError("depth must be positive")
    turns = trajectories.turn_ids
    valid = trajectories.response_mask & torch.ge(turns, 0)
    if not valid.any().item():
        raise ValueError("TCOD found no response turns")
    if direction == "f2b":
        return valid & torch.lt(turns, depth)
    if direction == "b2f":
        maximum_turn = torch.where(valid, turns, torch.full_like(turns, -1)).amax(
            dim=1, keepdim=True
        )
        retur

### 다른 선택지는 없나?

F2B는 초기 decision 오류가 핵심일 때, B2F는 긴 horizon 때문에 성공 경험이 희소할 때 자연스럽다. random window나 difficulty curriculum도 대안이지만 TCOD 결과로 부르려면 논문의 방향·prefix 조건을 보존해야 한다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L09의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
from opd_study.algorithms.tcod import curriculum_depth, temporal_curriculum_mask
from opd_study.data import CharacterTokenizer, collate_multiturn_text, generate_tiny_arithmetic

tokenizer = CharacterTokenizer(); rows = generate_tiny_arithmetic(train_rows=2, validation_rows=1, test_rows=1).train
batch = collate_multiturn_text([(row.prompt, tuple(row.response.splitlines())) for row in rows], tokenizer)
schedule = [curriculum_depth(step, start_depth=1, pacing_steps=2, maximum_depth=3)
            for step in range(6)]
print("depth schedule:", schedule)
for direction in ("f2b", "b2f"):
    mask = temporal_curriculum_mask(batch, depth=1, direction=direction)
    print(direction, "selected tokens:", int(mask.sum()))

depth schedule: [1, 1, 2, 2, 3, 3]
f2b selected tokens: 30
b2f selected tokens: 24


In [4]:
early = temporal_curriculum_mask(batch, depth=1, direction="f2b")
late = temporal_curriculum_mask(batch, depth=1, direction="b2f")
print("F2B learns beginnings; B2F needs a successful/teacher prefix before this suffix mask.")
print("overlap between one-turn windows:", int((early & late).sum()))

F2B learns beginnings; B2F needs a successful/teacher prefix before this suffix mask.
overlap between one-turn windows: 0


## Checks

In [5]:
assert schedule == [1, 1, 2, 2, 3, 3]
assert not (early & late).any()
assert batch.turn_ids is not None
print("check passed: pacing and temporal slices are explicit")

check passed: pacing and temporal slices are explicit


**연습 (8분):** 3-turn batch에서 depth 1/2/3의 F2B와 B2F mask를 표로 만들고 각 turn이 처음 포함되는 step을 적어라.

<details><summary>확인 기준</summary>F2B는 작은 turn ID부터, B2F는 큰 turn ID부터 포함하며 pacing과 depth를 구분한다.</details>

## 내가 자주 틀리는 것

### M1 — B2F를 단순 reverse mask로 구현하기

- 틀린 형태: 실패한 student prefix 뒤 suffix만 선택한다.
- 왜 틀렸나: B2F는 성공/teacher prefix 조건이 핵심이다.
- 고친 형태: prefix provenance와 선택 turn을 함께 기록한다.
- 관련 검사: `test_tcod_curriculum_and_directions`

### M2 — depth와 global step을 같은 값으로 쓰기

- 틀린 형태: 매 step마다 무조건 turn 하나를 늘린다.
- 왜 틀렸나: pacing schedule이 사라진다.
- 고친 형태: start depth, pacing steps, max depth를 명시한다.
- 관련 검사: `test_tcod_curriculum_and_directions`

## 60초 요약

1. 시간 깊이 schedule을 만든다
2. F2B/B2F를 구분한다
3. 선택된 turn을 감사한다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`tcod`](https://arxiv.org/abs/2604.24005v3) · `2604.24005v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`tcod_official`](https://github.com/kokolerk/TCOD) · `465eef4406ad0cff675b36bd46f37f28b1736ff9` · license `Apache-2.0` · [audited manifest](../../docs/sources.yml)